# Clustering agroclimático de cultivos (Pareto-80)

**Objetivo:** agrupar las 30 combinaciones `(región, cultivo)` del dataset integrado según su perfil agroclimático y productivo (2020–2025).

| Campo | Valor |
|-------|-------|
| Input | `OUTPUTS/dataset_integrado.csv` |
| Unidad principal | 30 perfiles `(región, cultivo)` |
| Análisis complementario | 2.160 observaciones mensuales (solo clima) |

**Decisiones metodológicas:**
- No imputar NaN de producción con cero
- Sin remoción de outliers (IQR/Z-score)
- Métricas DBSCAN: Silhouette calculada excluyendo puntos de ruido (`-1`)


## 1. Instalación e importaciones

In [ ]:
# =============================================================================
# PASO 1: Importar librerías
# =============================================================================
# Cada bloque cubre una parte del pipeline de clustering.

import warnings
from pathlib import Path  # Rutas multiplataforma (Windows / Colab / Linux)

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Clustering y métricas ---
from scipy.cluster.hierarchy import dendrogram, linkage  # Dendrograma jerárquico
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA, NMF  # Reducción de dimensionalidad
from sklearn.metrics import (
    calinski_harabasz_score,   # Índice Calinski-Harabasz (mayor = mejor separación)
    davies_bouldin_score,      # Davies-Bouldin (menor = clusters más compactos)
    silhouette_samples,
    silhouette_score,          # Silhouette [-1,1]: mayor = mejor agrupamiento
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler  # Escalado de features

warnings.filterwarnings("ignore")  # Ocultar avisos menores de sklearn/matplotlib
plt.rcParams["figure.dpi"] = 120   # Resolución de gráficos
sns.set_style("whitegrid")         # Estilo visual consistente
PALETTE = sns.color_palette("tab10")  # Colores para gráficos multi-cluster

print("Librerías cargadas")


## 2. Carga y exploración

In [ ]:
# =============================================================================
# PASO 2: Configurar rutas y cargar el dataset integrado
# =============================================================================

# Detectar la raíz del proyecto: si ejecutas desde Clustering/, sube un nivel
ROOT = Path.cwd()
if ROOT.name == "Clustering":
    ROOT = ROOT.parent

# Rutas de entrada/salida
RUTA_DATA = ROOT / "OUTPUTS" / "dataset_integrado.csv"  # Tabla maestra Pareto-80
RUTA_OUT = ROOT / "OUTPUTS"                              # CSVs exportados
RUTA_FIG = RUTA_OUT / "figures"                          # Figuras para el paper
RUTA_FIG.mkdir(parents=True, exist_ok=True)

# Cargar datos: 2.160 filas = 30 (región,cultivo) × 72 meses (2020-2025)
df = pd.read_csv(RUTA_DATA)

# Diagnóstico rápido de dimensiones
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Combinaciones (región, cultivo): {df.groupby(['region', 'cultivo']).ngroups}")
print("Columnas:", list(df.columns))
df.head()


In [ ]:
# =============================================================================
# PASO 2b: Definir variables climáticas y revisar cobertura
# =============================================================================

# Las 12 variables NASA exportadas con nombres legibles en español
CLIMA_VARS = [
    "temp_promedio", "temp_maxima", "temp_minima", "precipitacion",
    "humedad_relativa", "radiacion_solar", "velocidad_viento",
    "presion_atmosferica", "humedad_suelo", "temp_superficie",
    "punto_rocio", "humedad_especifica",
]

# Subconjunto para el perfil agroclimático (evita redundancia temp_max/min)
CLIMA_CORE = [
    "temp_promedio", "precipitacion", "humedad_relativa",
    "radiacion_solar", "humedad_suelo",
]

print("Regiones :", sorted(df["region"].unique()))
print("Pisos    :", sorted(df["piso_ecologico"].unique()))
print("Cultivos :", df["cultivo"].nunique())
# NaN en producción = meses sin dato MIDAGRI (NO se rellenan con cero)
print("NaN produccion_ton:", df["produccion_ton"].isna().sum())


## 3. Funciones auxiliares de clustering

Reutilizables en el análisis por perfiles y el análisis mensual complementario.


In [ ]:
# =============================================================================
# PASO 3: Funciones auxiliares reutilizables
# =============================================================================
# Centralizamos la lógica para no repetirla en perfiles y análisis mensual.

# Rango de K a probar en KMeans (perfiles: 2-7; mensual: se puede ampliar)
K_RANGE = range(2, 9)


def silhouette_no_noise(X, labels):
    """Silhouette excluyendo puntos marcados como ruido (-1) por DBSCAN."""
    mask = labels >= 0          # True = punto asignado a un cluster
    labs = labels[mask]
    if len(set(labs)) < 2 or mask.sum() < 3:
        return np.nan           # No se puede calcular con <2 clusters
    return float(silhouette_score(X[mask], labs))


def pct_ruido(labels):
    """Porcentaje de observaciones que DBSCAN clasificó como ruido (etiqueta -1)."""
    return 100.0 * (labels < 0).mean()


def eval_kmeans_range(X, k_range=K_RANGE, random_state=42):
    """Entrena KMeans para cada K y devuelve métricas de calidad."""
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        lbl = km.fit_predict(X)
        rows.append({
            "k": k,
            "inertia": km.inertia_,  # Suma distancias² al centroide (método del codo)
            "silhouette": silhouette_score(X, lbl),
            "calinski": calinski_harabasz_score(X, lbl),
            "davies_bouldin": davies_bouldin_score(X, lbl),
            "labels": lbl,
            "model": km,
        })
    return pd.DataFrame(rows)


def elegir_k(df_km):
    """Elige K óptimo combinando Silhouette (max) y Davies-Bouldin (min)."""
    best_sil = int(df_km.loc[df_km["silhouette"].idxmax(), "k"])
    best_db = int(df_km.loc[df_km["davies_bouldin"].idxmin(), "k"])
    if best_sil == best_db:
        return best_sil, "coincidencia Silhouette y Davies-Bouldin"
    # Si discrepan: promediar rankings de ambas métricas
    rank_sil = df_km.set_index("k")["silhouette"].rank(ascending=False)
    rank_db = df_km.set_index("k")["davies_bouldin"].rank(ascending=True)
    k_opt = int((rank_sil + rank_db).idxmin())
    return k_opt, f"ranking promedio (Sil={best_sil}, DB={best_db})"


def grid_dbscan(X, eps_list, ms_list):
    """Barrido de hiperparámetros DBSCAN: eps (radio) y min_samples (densidad mínima)."""
    rows = []
    for eps in eps_list:
        for ms in ms_list:
            lbl = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            n_cl = len(set(lbl)) - (1 if -1 in lbl else 0)
            if n_cl < 2:
                continue  # Silhouette requiere al menos 2 clusters
            rows.append({
                "eps": eps,
                "min_samples": ms,
                "n_clusters": n_cl,
                "pct_ruido": pct_ruido(lbl),
                "silhouette": silhouette_no_noise(X, lbl),
                "davies_bouldin": davies_bouldin_score(X[lbl >= 0], lbl[lbl >= 0]),
                "labels": lbl,
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("silhouette", ascending=False)


def plot_k_selection(df_km, titulo):
    """Gráfico triple: codo (inercia), Silhouette y Davies-Bouldin vs K."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df_km["k"], df_km["inertia"], "o-", color="steelblue")
    axes[0].set_title("Codo (inercia)")
    axes[0].set_xlabel("K")
    best_sil = int(df_km.loc[df_km["silhouette"].idxmax(), "k"])
    axes[1].plot(df_km["k"], df_km["silhouette"], "o-", color="green")
    axes[1].axvline(best_sil, color="red", ls="--", label=f"K={best_sil}")
    axes[1].set_title("Silhouette")
    axes[1].legend()
    best_db = int(df_km.loc[df_km["davies_bouldin"].idxmin(), "k"])
    axes[2].plot(df_km["k"], df_km["davies_bouldin"], "o-", color="crimson")
    axes[2].axvline(best_db, color="navy", ls="--", label=f"K={best_db}")
    axes[2].set_title("Davies-Bouldin")
    axes[2].legend()
    plt.suptitle(titulo, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 4. Análisis principal — perfiles por (región, cultivo)

Una fila por combinación Pareto-80 (~30). Agrega clima (media y desv. estándar) y producción sin imputar ceros en meses faltantes.


In [ ]:
# =============================================================================
# PASO 4: Construir perfiles agroclimáticos (1 fila por región × cultivo)
# =============================================================================
# Pasamos de 2.160 filas mensuales → ~30 perfiles interpretables para el paper.

def coef_var_positivos(s):
    """Coeficiente de variación solo en meses con producción > 0 (cosecha real)."""
    v = s.dropna()
    v = v[v > 0]
    if len(v) < 2 or v.mean() == 0:
        return np.nan
    return v.std() / v.mean()

# Diccionario de agregaciones: (columna_origen, función)
# IMPORTANTE: dropna() y filtros evitan convertir NaN en cero
agg_dict = {
    "produccion_total": ("produccion_ton", lambda s: s.dropna().sum()),
    "produccion_media_cosecha": (
        "produccion_ton",
        lambda s: s[s > 0].mean() if (s > 0).any() else np.nan,
    ),
    "coef_var_produccion": ("produccion_ton", coef_var_positivos),
    "n_meses_con_dato": ("produccion_ton", lambda s: s.notna().sum()),
    "n_meses_cosecha": ("produccion_ton", lambda s: (s > 0).sum()),
}
# Para cada variable climática clave: media y desviación estándar 2020-2025
for v in CLIMA_CORE:
    agg_dict[f"{v}_mean"] = (v, "mean")
    agg_dict[f"{v}_std"] = (v, "std")

# Agrupar por la unidad de análisis principal
df_perfil = df.groupby(
    ["region", "cultivo", "piso_ecologico", "distrito"], as_index=False
).agg(**agg_dict)

# Etiqueta legible para gráficos (dendrograma, anotaciones PCA)
df_perfil["etiqueta"] = df_perfil["region"] + " | " + df_perfil["cultivo"]

# Features numéricas que entrarán al clustering (sin metadatos categóricos)
FEATURES_PERFIL = [c for c in df_perfil.columns if c.endswith("_mean") or c.endswith("_std")]
FEATURES_PERFIL += [
    "produccion_total", "produccion_media_cosecha", "coef_var_produccion",
]

# Escalar a media=0 y std=1 para que KMeans no domine por magnitud (ej. producción en ton)
X_perfil_raw = df_perfil[FEATURES_PERFIL].copy()
scaler_perfil = StandardScaler()
X_perfil = scaler_perfil.fit_transform(X_perfil_raw)

print(f"Perfiles: {df_perfil.shape[0]} filas x {len(FEATURES_PERFIL)} features")
df_perfil[["region", "cultivo", "produccion_total", "n_meses_cosecha"]].head(10)


In [ ]:
# =============================================================================
# PASO 4a: KMeans sobre perfiles (método principal recomendado para el paper)
# =============================================================================

# 1) Probar K=2..7 y graficar métricas de selección
km_perfil_df = eval_kmeans_range(X_perfil, k_range=range(2, 8))
K_PERFIL, motivo_k = elegir_k(km_perfil_df)
plot_k_selection(km_perfil_df, f"Selección de K — perfiles (K={K_PERFIL})")
print(f"K óptimo perfiles: {K_PERFIL} ({motivo_k})")

# 2) Entrenar modelo final con el K elegido
km_perfil = KMeans(n_clusters=K_PERFIL, random_state=42, n_init=10)
labels_perfil_km = km_perfil.fit_predict(X_perfil)

# 3) Guardar etiquetas en el DataFrame de perfiles
df_perfil["cluster_kmeans"] = labels_perfil_km

# 4) Evaluar calidad del agrupamiento
sil_perfil_km = silhouette_score(X_perfil, labels_perfil_km)
db_perfil_km = davies_bouldin_score(X_perfil, labels_perfil_km)
print(f"Silhouette={sil_perfil_km:.4f}  Davies-Bouldin={db_perfil_km:.4f}")


In [ ]:
# =============================================================================
# PASO 4b: Clustering jerárquico + dendrograma
# =============================================================================
# Útil para visualizar similitud entre cultivos y elegir K de forma cualitativa.

# Matriz de enlaces con método Ward (minimiza varianza intra-cluster)
Z = linkage(X_perfil, method="ward")

# Dendrograma: altura del enlace = disimilitud; cortes horizontales = clusters
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z,
    labels=df_perfil["etiqueta"].tolist(),
    leaf_rotation=90,
    leaf_font_size=8,
    color_threshold=None,
    ax=ax,
)
ax.set_title("Dendrograma — perfiles (región | cultivo)", fontweight="bold")
plt.tight_layout()
plt.savefig(RUTA_FIG / "dendrograma_perfiles.png", dpi=150, bbox_inches="tight")
plt.show()

# Mismo K que KMeans para comparar métodos partitivos vs jerárquicos
hc = AgglomerativeClustering(n_clusters=K_PERFIL, linkage="ward")
labels_perfil_hc = hc.fit_predict(X_perfil)
df_perfil["cluster_jerarquico"] = labels_perfil_hc
sil_perfil_hc = silhouette_score(X_perfil, labels_perfil_hc)
print(f"Jerárquico K={K_PERFIL} | Silhouette={sil_perfil_hc:.4f}")


In [ ]:
# =============================================================================
# PASO 4c: DBSCAN sobre perfiles (clusters de forma arbitraria + detección de ruido)
# =============================================================================
# Grid más pequeño porque solo hay ~30 puntos (sensibles a eps y min_samples).

res_db_perfil = grid_dbscan(
    X_perfil,
    eps_list=[0.8, 1.0, 1.2, 1.5, 2.0, 2.5],  # Radio de vecindad en espacio escalado
    ms_list=[2, 3, 4],                          # Mínimo de vecinos para núcleo denso
)

if len(res_db_perfil):
    best_db_p = res_db_perfil.iloc[0]  # Mejor Silhouette (sin ruido)
    labels_perfil_db = best_db_p["labels"].astype(int)
    df_perfil["cluster_dbscan"] = labels_perfil_db
    sil_perfil_db = best_db_p["silhouette"]
    pct_noise_p = best_db_p["pct_ruido"]
    print("Top DBSCAN perfiles:")
    print(res_db_perfil[["eps", "min_samples", "n_clusters", "pct_ruido", "silhouette"]].head(5).to_string(index=False))
else:
    # Ninguna combinación produjo ≥2 clusters
    labels_perfil_db = np.full(len(df_perfil), -1)
    df_perfil["cluster_dbscan"] = labels_perfil_db
    sil_perfil_db = np.nan
    pct_noise_p = 100.0
    print("DBSCAN perfiles: sin configuración válida")


### 4.1 Interpretación de clusters (perfiles)

In [ ]:
# =============================================================================
# PASO 4d: Interpretar clusters — ¿qué cultivos quedaron juntos?
# =============================================================================

# Tabla exportable: asignación por método
tabla_perfil = df_perfil[
    ["region", "cultivo", "piso_ecologico", "distrito",
     "cluster_kmeans", "cluster_jerarquico", "cluster_dbscan", "produccion_total"]
].sort_values(["cluster_kmeans", "region", "cultivo"])
print(tabla_perfil.to_string(index=False))

# ¿Los clusters mezclan regiones o respetan geografía?
comp_region = pd.crosstab(df_perfil["cluster_kmeans"], df_perfil["region"])
print("\nComposición KMeans por región:")
print(comp_region)

# Heatmap: perfil medio de cada cluster (normalizado 0-1 para comparar variables)
cent = df_perfil.groupby("cluster_kmeans")[FEATURES_PERFIL].mean()
cent_norm = (cent - cent.min()) / (cent.max() - cent.min() + 1e-9)
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(cent_norm.T, annot=False, cmap="YlOrRd", ax=ax)
ax.set_title("Perfil normalizado por cluster KMeans", fontweight="bold")
ax.set_xticklabels([f"C{i}" for i in cent.index])
plt.tight_layout()
plt.savefig(RUTA_FIG / "heatmap_perfiles_kmeans.png", dpi=150, bbox_inches="tight")
plt.show()

# PCA 2D solo para visualización (no para entrenar el cluster)
pca_p = PCA(n_components=2, random_state=42)
xy_p = pca_p.fit_transform(X_perfil)
fig, ax = plt.subplots(figsize=(9, 7))
for c in sorted(df_perfil["cluster_kmeans"].unique()):
    m = df_perfil["cluster_kmeans"] == c
    ax.scatter(xy_p[m, 0], xy_p[m, 1], label=f"Cluster {c}", s=80, alpha=0.85)
for idx, (_, row) in enumerate(df_perfil.iterrows()):
    ax.annotate(row["cultivo"][:12], (xy_p[idx, 0], xy_p[idx, 1]), fontsize=6, alpha=0.7)
ax.set_title("PCA perfiles — KMeans", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Análisis complementario — observaciones mensuales (solo clima)

> **Limitación:** cada `(región, cultivo)` aporta 72 filas con clima casi idéntico. Las métricas de Silhouette pueden verse infladas por repetición temporal. Se usa **solo variables climáticas** (sin `fillna(0)` en producción).


In [ ]:
# =============================================================================
# PASO 5: Análisis complementario — filas mensuales (SOLO clima)
# =============================================================================
# ADVERTENCIA: 72 filas por cultivo repiten el mismo clima → Silhouette puede inflarse.
# No usamos produccion_ton ni fillna(0).

# Quedarnos solo con meses que tienen las 12 variables climáticas completas
df_clima = df.dropna(subset=CLIMA_VARS, how="any").copy()
X_clima_raw = df_clima[CLIMA_VARS].values

# Escalado estándar para KMeans/DBSCAN/PCA
scaler_clima = StandardScaler()
X_clima = scaler_clima.fit_transform(X_clima_raw)

# MinMax [0,1] para NMF (requiere valores no negativos)
scaler_mm = MinMaxScaler()
X_clima_mm = scaler_mm.fit_transform(X_clima_raw)

print(f"Filas mensuales con clima completo: {len(df_clima):,}")


### 5.1 KMeans y DBSCAN (mensual, clima)

In [ ]:
# =============================================================================
# PASO 5a: KMeans y DBSCAN en espacio climático mensual
# =============================================================================

km_mensual_df = eval_kmeans_range(X_clima, k_range=range(2, 11))
K_MENSUAL, _ = elegir_k(km_mensual_df)
plot_k_selection(km_mensual_df, f"KMeans mensual — solo clima (K={K_MENSUAL})")

km_mensual = KMeans(n_clusters=K_MENSUAL, random_state=42, n_init=10)
labels_mensual_km = km_mensual.fit_predict(X_clima)
sil_mensual_km = silhouette_score(X_clima, labels_mensual_km)
db_mensual_km = davies_bouldin_score(X_clima, labels_mensual_km)
print(f"KMeans mensual K={K_MENSUAL} | Sil={sil_mensual_km:.4f}")

# DBSCAN con grid más amplio (más puntos que en perfiles)
res_db_mensual = grid_dbscan(
    X_clima,
    eps_list=[0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0],
    ms_list=[3, 5, 10, 15],
)
if len(res_db_mensual):
    best_db_m = res_db_mensual.iloc[0]
    labels_mensual_db = best_db_m["labels"].astype(int)
    sil_mensual_db = best_db_m["silhouette"]
    pct_noise_m = best_db_m["pct_ruido"]
    print(f"DBSCAN mensual eps={best_db_m['eps']} ms={int(best_db_m['min_samples'])} | Sil={sil_mensual_db:.4f} | ruido={pct_noise_m:.1f}%")
else:
    labels_mensual_db = np.full(len(df_clima), -1)
    sil_mensual_db = np.nan
    pct_noise_m = 100.0


### 5.2 PCA + clustering mensual

In [ ]:
# =============================================================================
# PASO 5b: PCA — reducir 12 variables climáticas y clusterizar en espacio reducido
# =============================================================================

# Ajustar PCA completo para ver varianza explicada por cada componente
pca_full = PCA(random_state=42)
pca_full.fit(X_clima)
var_ac = np.cumsum(pca_full.explained_variance_ratio_)

# Elegir número de componentes que acumulan ≥90% de varianza (máx 8)
n_pc_90 = int(np.searchsorted(var_ac, 0.90) + 1)
n_pc_90 = max(2, min(n_pc_90, 8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(var_ac) + 1), var_ac, "o-")
axes[0].axhline(0.9, ls="--", color="gray")
axes[0].set_title("Varianza acumulada PCA")
# Loadings: peso de cada variable climática en PC1 y PC2
load = pd.DataFrame(pca_full.components_[:2].T, index=CLIMA_VARS, columns=["PC1", "PC2"])
sns.heatmap(load, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Loadings PC1–PC2")
plt.tight_layout()
plt.show()

# Proyectar datos y aplicar KMeans + DBSCAN en el espacio PCA
pca = PCA(n_components=n_pc_90, random_state=42)
X_pca = pca.fit_transform(X_clima)
km_pca_df = eval_kmeans_range(X_pca, k_range=range(2, 11))
K_PCA, _ = elegir_k(km_pca_df)
labels_pca_km = KMeans(n_clusters=K_PCA, random_state=42, n_init=10).fit_predict(X_pca)
sil_pca_km = silhouette_score(X_pca, labels_pca_km)

res_pca_db = grid_dbscan(X_pca, eps_list=[0.8, 1.2, 1.5, 2.0, 2.5], ms_list=[5, 10, 15, 20])
if len(res_pca_db):
    best_pca_db = res_pca_db.iloc[0]
    labels_pca_db = best_pca_db["labels"].astype(int)
    sil_pca_db = best_pca_db["silhouette"]
    pct_noise_pca = best_pca_db["pct_ruido"]
else:
    labels_pca_db = np.full(len(X_pca), -1)
    sil_pca_db = np.nan
    pct_noise_pca = 100.0
print(f"PCA ({n_pc_90} PCs) + KMeans K={K_PCA} | Sil={sil_pca_km:.4f}")


### 5.3 NMF + clustering mensual

In [ ]:
# =============================================================================
# PASO 5c: NMF — factores latentes no negativos + clustering
# =============================================================================
# NMF descompone X ≈ W·H con W,H ≥ 0 → perfiles interpretables como "mezclas" de patrones.

nmf_errors = []
for n in range(2, 10):
    nmf_tmp = NMF(n_components=n, random_state=42, max_iter=500).fit(X_clima_mm)
    nmf_errors.append(nmf_tmp.reconstruction_err_)

# Elegir N donde el error de reconstrucción cae más (codo en la curva de error)
drops = np.diff(nmf_errors)
N_NMF = int(np.argmin(drops) + 2) if len(drops) else 3
N_NMF = max(2, min(N_NMF, 8))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(2, 10), nmf_errors, "o-", color="purple")
ax.axvline(N_NMF, ls="--", color="red", label=f"N={N_NMF}")
ax.set_title("Error reconstrucción NMF")
ax.legend()
plt.tight_layout()
plt.show()

# Matriz W: cada fila mensual como combinación de N factores latentes
nmf = NMF(n_components=N_NMF, random_state=42, max_iter=500)
X_nmf = nmf.fit_transform(X_clima_mm)

km_nmf_df = eval_kmeans_range(X_nmf, k_range=range(2, 11))
K_NMF, _ = elegir_k(km_nmf_df)
labels_nmf_km = KMeans(n_clusters=K_NMF, random_state=42, n_init=10).fit_predict(X_nmf)
sil_nmf_km = silhouette_score(X_nmf, labels_nmf_km)

res_nmf_db = grid_dbscan(X_nmf, eps_list=[0.5, 1.0, 1.5, 2.0], ms_list=[5, 10, 15])
if len(res_nmf_db):
    best_nmf_db = res_nmf_db.iloc[0]
    labels_nmf_db = best_nmf_db["labels"].astype(int)
    sil_nmf_db = best_nmf_db["silhouette"]
    pct_noise_nmf = best_nmf_db["pct_ruido"]
else:
    labels_nmf_db = np.full(len(X_nmf), -1)
    sil_nmf_db = np.nan
    pct_noise_nmf = 100.0
print(f"NMF ({N_NMF}) + KMeans K={K_NMF} | Sil={sil_nmf_km:.4f}")


## 6. Comparativa final y exportación

In [ ]:
# =============================================================================
# PASO 6: Comparativa final de todos los métodos
# =============================================================================
# Unimos métricas de perfiles (interpretables) y mensuales (complementarios).

metricas = pd.DataFrame([
  # --- Análisis principal: perfiles (~30 unidades) ---
    {"configuracion": f"Perfil KMeans K={K_PERFIL}", "unidad": "perfil", "metodo": "KMeans",
     "reduccion": "Ninguna", "silhouette": sil_perfil_km, "davies_bouldin": db_perfil_km,
     "pct_ruido": 0.0, "interpretabilidad": "Alta"},
    {"configuracion": f"Perfil Jerárquico K={K_PERFIL}", "unidad": "perfil", "metodo": "Jerárquico",
     "reduccion": "Ninguna", "silhouette": sil_perfil_hc, "davies_bouldin": davies_bouldin_score(X_perfil, labels_perfil_hc),
     "pct_ruido": 0.0, "interpretabilidad": "Alta"},
    {"configuracion": "Perfil DBSCAN", "unidad": "perfil", "metodo": "DBSCAN",
     "reduccion": "Ninguna", "silhouette": sil_perfil_db, "davies_bouldin": np.nan,
     "pct_ruido": pct_noise_p if len(res_db_perfil) else 100.0, "interpretabilidad": "Alta"},
  # --- Análisis complementario: mensual (2.160 filas, solo clima) ---
    {"configuracion": f"Mensual KMeans K={K_MENSUAL}", "unidad": "mensual", "metodo": "KMeans",
     "reduccion": "Ninguna", "silhouette": sil_mensual_km, "davies_bouldin": db_mensual_km,
     "pct_ruido": 0.0, "interpretabilidad": "Baja"},
    {"configuracion": "Mensual DBSCAN", "unidad": "mensual", "metodo": "DBSCAN",
     "reduccion": "Ninguna", "silhouette": sil_mensual_db, "davies_bouldin": np.nan,
     "pct_ruido": pct_noise_m, "interpretabilidad": "Baja"},
    {"configuracion": f"PCA+KMeans K={K_PCA}", "unidad": "mensual", "metodo": "KMeans",
     "reduccion": "PCA", "silhouette": sil_pca_km, "davies_bouldin": np.nan,
     "pct_ruido": 0.0, "interpretabilidad": "Media"},
    {"configuracion": "PCA+DBSCAN", "unidad": "mensual", "metodo": "DBSCAN",
     "reduccion": "PCA", "silhouette": sil_pca_db, "davies_bouldin": np.nan,
     "pct_ruido": pct_noise_pca, "interpretabilidad": "Media"},
    {"configuracion": f"NMF+KMeans K={K_NMF}", "unidad": "mensual", "metodo": "KMeans",
     "reduccion": "NMF", "silhouette": sil_nmf_km, "davies_bouldin": np.nan,
     "pct_ruido": 0.0, "interpretabilidad": "Media"},
    {"configuracion": "NMF+DBSCAN", "unidad": "mensual", "metodo": "DBSCAN",
     "reduccion": "NMF", "silhouette": sil_nmf_db, "davies_bouldin": np.nan,
     "pct_ruido": pct_noise_nmf, "interpretabilidad": "Media"},
])
metricas[["silhouette", "davies_bouldin", "pct_ruido"]] = metricas[
    ["silhouette", "davies_bouldin", "pct_ruido"]
].round(4)
print(metricas.to_string(index=False))

# Recomendación automática: mejor Silhouette entre métodos de PERFIL (alta interpretabilidad)
cand = metricas[(metricas["unidad"] == "perfil") & metricas["silhouette"].notna()]
recomendado = cand.loc[cand["silhouette"].idxmax(), "configuracion"]
print(f"\nRecomendado para el paper: {recomendado}")


In [ ]:
# Gráfico comparativo: azul = perfiles, gris = mensual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
short = metricas["configuracion"].str.replace(" ", "\n", n=1)
colors = ["#2196F3" if u == "perfil" else "#9E9E9E" for u in metricas["unidad"]]

axes[0].bar(range(len(metricas)), metricas["silhouette"], color=colors)
axes[0].set_xticks(range(len(metricas)))
axes[0].set_xticklabels(short, fontsize=7, rotation=45, ha="right")
axes[0].set_title("Silhouette (mayor = mejor)")

axes[1].bar(range(len(metricas)), metricas["pct_ruido"], color=colors)
axes[1].set_xticks(range(len(metricas)))
axes[1].set_xticklabels(short, fontsize=7, rotation=45, ha="right")
axes[1].set_title("% ruido DBSCAN")

plt.suptitle("Comparativa perfiles vs mensual", fontweight="bold")
plt.tight_layout()
plt.savefig(RUTA_FIG / "comparativa_clustering.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# =============================================================================
# PASO 6b: Exportar resultados para el paper y otros notebooks
# =============================================================================

export_perfil = tabla_perfil.copy()
export_perfil.to_csv(RUTA_OUT / "clustering_perfiles.csv", index=False, encoding="utf-8-sig")
metricas.to_csv(RUTA_OUT / "clustering_metricas.csv", index=False, encoding="utf-8-sig")
print("Exportado:")
print(f"  {RUTA_OUT / 'clustering_perfiles.csv'}")
print(f"  {RUTA_OUT / 'clustering_metricas.csv'}")


## 7. Conclusiones

In [ ]:
# =============================================================================
# PASO 7: Conclusiones — interpretación agronómica (no solo métricas)
# =============================================================================

print("=" * 60)
print("CONCLUSIONES")
print("=" * 60)
print(f"Método recomendado: {recomendado}")
print(f"Perfiles analizados: {len(df_perfil)} combinaciones (región, cultivo)")
print()

# Listar cultivos que comparten cada cluster KMeans
for c in sorted(df_perfil["cluster_kmeans"].unique()):
    sub = df_perfil[df_perfil["cluster_kmeans"] == c]
    cultivos = ", ".join(sub["region"] + "-" + sub["cultivo"])
    pisos = ", ".join(sorted(sub["piso_ecologico"].unique()))
    print(f"Cluster KMeans {c} ({len(sub)} cultivos):")
    print(f"  Cultivos: {cultivos[:200]}{'...' if len(cultivos)>200 else ''}")
    print(f"  Pisos: {pisos}")
    print()

print("Hallazgos:")
print("- El clustering por PERFIL agrupa cultivos con condiciones agroclimáticas similares.")
print("- El análisis MENSUAL infla Silhouette por 72 réplicas por cultivo; usar solo como complemento.")
print("- No se imputaron NaN de producción; coeficientes usan solo meses con dato real.")
print("- Limitación: 30 unidades — correlación no implica causalidad clima→producción.")
